### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [3]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encode_geography.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('Scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [4]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [5]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [6]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [7]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/sankalpmakol/Desktop/ANN CLASSIFICATION/.venv/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sankalpmakol/Desktop/ANN CLASSIFICATION/.venv/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sankalpmakol/Desktop/ANN CLASSIFICATION/.venv/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/sankalpmakol/Desktop/ANN CLASSIFICATION/.venv/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


2026-07-04 20:37:43.026360: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:43.084404: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:44.746173: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:44.762013: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:44.795347: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:44.805679: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-07-04 20:37:44.842850: I tensorflow/core/grappler/optimizers/cust

167/167 [==============================] - 15s 28ms/step - loss: 0.5705 - accuracy: 0.7041
Epoch 2/50
167/167 [==============================] - 15s 28ms/step - loss: 0.5117 - accuracy: 0.7725
Epoch 2/50
167/167 [==============================] - 15s 24ms/step - loss: 0.7160 - accuracy: 0.5944
Epoch 2/50
Epoch 2/50
167/167 [==============================] - 15s 24ms/step - loss: 0.5677 - accuracy: 0.7195
Epoch 2/50
167/167 [==============================] - 15s 24ms/step - loss: 0.5430 - accuracy: 0.7352
Epoch 2/50
167/167 [==============================] - 3s 16ms/step - loss: 0.4487 - accuracy: 0.8056
Epoch 3/50
167/167 [==============================] - 3s 16ms/step - loss: 0.4390 - accuracy: 0.8082
Epoch 3/50
167/167 [==============================] - 3s 15ms/step - loss: 0.4636 - accuracy: 0.8063
Epoch 3/50
167/167 [==============================] - 3s 15ms/step - loss: 0.4940 - accuracy: 0.7950
Epoch 3/50
167/167 [==============================] - 3s 16ms/step - loss: 0.4350 - ac

...

167/167 [==============================] - 3s 17ms/step - loss: 0.4528 - accuracy: 0.8022
Epoch 10/100
167/167 [==============================] - 3s 16ms/step - loss: 0.4610 - accuracy: 0.8057
Epoch 11/100
167/167 [==============================] - 3s 18ms/step - loss: 0.4957 - accuracy: 0.8005
Epoch 10/100
167/167 [==============================] - 3s 18ms/step - loss: 0.4376 - accuracy: 0.8035
Epoch 3/100
167/167 [==============================] - 3s 19ms/step - loss: 0.4459 - accuracy: 0.8057
Epoch 3/100
167/167 [==============================] - 3s 18ms/step - loss: 0.5158 - accuracy: 0.7939
Epoch 10/100
167/167 [==============================] - 3s 19ms/step - loss: 0.4606 - accuracy: 0.8078
Epoch 12/100
167/167 [==============================] - 3s 17ms/step - loss: 0.4589 - accuracy: 0.8016
Epoch 4/100
167/167 [==============================] - 3s 16ms/step - loss: 0.5346 - accuracy: 0.7939
Epoch 11/100
167/167 [==============================] - 3s 17ms/step - loss: 0.4670 - acc

/Users/sankalpmakol/Desktop/ANN CLASSIFICATION/.venv/lib/python3.11/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
2026-07-04 20:59:53.253555: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-07-04 20:59:53.269193: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-07-04 20:59:53.270205: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-07-04 20:59:53.270317: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-04 20:59:53.270660: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 wit

Epoch 1/50


2026-07-04 20:59:55.132877: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


250/250 [==============================] - 4s 4ms/step - loss: 0.6201 - accuracy: 0.6915
Epoch 2/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4434 - accuracy: 0.8129
Epoch 3/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4340 - accuracy: 0.8127
Epoch 4/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4334 - accuracy: 0.8124
Epoch 5/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4334 - accuracy: 0.8099
Epoch 6/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4335 - accuracy: 0.8123
Epoch 7/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4334 - accuracy: 0.8121
Epoch 8/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4333 - accuracy: 0.8112
Epoch 9/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4331 - accuracy: 0.8133
Epoch 10/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4332 - accuracy: 0.8098
Epoch 11/5